In [ ]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [ ]:
q("""Select * from customers;""")

In [ ]:
q("""     
select count(*) as total_count,
count(DISTINCT circle_id) as distinct_count
from circles
;""")

In [ ]:
q("""     
select count(*) as total_count,
count(DISTINCT plan_id) as distinct_count
from plans
;""")

In [ ]:
q("""     
select count(*) as total_count,
count(DISTINCT usage_id) as distinct_count
from usage_daily
;""")

In [ ]:
q("""     
select count(*) as total_count,
count(DISTINCT recharge_id) as distinct_count
from recharges
;""")

In [ ]:
q("""     
select count(*) as total_count,
count(DISTINCT complaint_id) as distinct_count
from complaints
;""")

In [ ]:
q("""select count(*)
     from circles c
     join customers cu
     on cu.circle_id=c.circle_id;""")

In [ ]:
q("""select count(*)
     from circles c
     left join customers cu
     on cu.currend=c.circle_id;""")

In [ ]:
q("""select count(*)
     from plans p
     join customers cu
     on cu.current_plan_id=p.plan_id;""")

In [ ]:
q("""select count(*)
     from recharges r
     join customers cu
     on cu.customer_id=r.customer_id;""")

In [ ]:
q("""select count(*)
     from recharges r
     left join customers cu
     on cu.customer_id=r.customer_id;""")

In [ ]:
q("""select count(*)
     from recharges r
     join plans p
     on p.plan_id=r.plan_id;""")

In [ ]:
q("""select count(*)
     from recharges r
     left join plans p
     on p.plan_id=r.plan_id;""")

In [ ]:
q("""select count(*)
     from usage_daily ud
     join customers cu
     on cu.customer_id=ud.customer_id;""")

In [ ]:
q("""select count(*)
     from usage_daily ud
     left join customers cu
     on cu.customer_id=ud.customer_id;""")

In [ ]:
q("""
 select * from usage_daily
 """)

In [ ]:
q(""" select * from customers""")

In [ ]:
q(""" select sum(is_churned=1 and churn_date is not null)
         as left_dated,
         sum(is_churned=0 and churn_date is null)
         as stayed_blank
from customers;""")

In [ ]:
q("""
SELECT SUM(is_resolved = 1 AND resolution_days IS NOT NULL) 
            AS left_dated,
            SUM(is_resolved = 0 AND resolution_days IS NULL) 
            AS stayed_blank
     FROM complaints;
""")

In [ ]:
q("""
SELECT
    c.customer_id,
    c.connection_type,
    (
        SELECT COUNT(*)
        FROM recharges r
        WHERE r.customer_id = c.customer_id
    ) AS payments
FROM customers c
WHERE NOT EXISTS (
    SELECT 1
    FROM usage_daily u
    WHERE u.customer_id = c.customer_id
);
""")

In [ ]:
q("""WITH l AS (SELECT customer_id,     
             MAX(usage_date) AS last_use         
           FROM usage_daily GROUP BY customer_id)
SELECT c.is_churned, COUNT(*) AS subs, 
       ROUND(AVG(julianday('2025-06-30')
             - julianday(l.last_use)), 2)     
         AS avg_quiet,                    
       MIN(julianday('2025-06-30')              
             - julianday(l.last_use)) AS min_,
       MAX(julianday('2025-06-30')              
             - julianday(l.last_use)) AS max_   
FROM   customers c JOIN l                
       ON l.customer_id = c.customer_id     
GROUP  BY c.is_churned;""")

In [ ]:
q("""SELECT 'usage_daily.usage_date' AS col,        
       MIN(usage_date) AS first_,                
       MAX(usage_date) AS last_                
FROM   usage_daily                    
UNION ALL SELECT 'customers.activation_date',   
       MIN(activation_date), MAX(activation_date)
FROM   customers                              
UNION ALL SELECT 'customers.churn_date',
       MIN(churn_date), MAX(churn_date) 
FROM   customers                      
UNION ALL SELECT 'recharges.recharge_date', 
       MIN(recharge_date), MAX(recharge_date)
FROM   recharges                           
UNION ALL SELECT 'complaints.complaint_date',
       MIN(complaint_date), MAX(complaint_date)
FROM   complaints;""")


In [ ]:
q("""WITH t AS (SELECT customer_id,                
             SUM(data_mb) AS total,           
             COUNT(*) AS days_active         
           FROM usage_daily GROUP BY customer_id) 
SELECT ROUND(AVG(total / days_active), 2)      
         AS per_active_day,                    
       ROUND(AVG(total / 181.0), 2)              
         AS per_calendar_day              
FROM   t;""")